In [1]:
import sys
sys.path.append('..')

import os
import re
import glob
import pandas as pd

from nnspike.utils import extract_video_frames
from nnspike.data import create_label_dataframe, sort_by_frames_number, label_dataset_by_opencv, label_dataset_by_model, augment_dataset, set_spike_status
from nnspike.constants import  ROI_CNN

course = "right" # "right" or "left"

## Extract Frames from Videos

In [2]:
def get_all_avi_files(directory_path="C:/Users/MSAD/github/nnspike/storage/20250824/videos/", filter_timestamp=None):
    """
    Get all AVI files from the specified directory with their timestamps.
    
    Args:
        directory_path (str): Path to the directory containing AVI files
        filter_timestamp (str, optional): If set, only return files matching this timestamp pattern (supports wildcards with *)
    
    Returns:
        list: List of tuples containing (file_path, timestamp)
    """
    import os
    import glob
    import re
    import fnmatch

    # Use glob to find all .avi files in the directory
    avi_files = glob.glob(os.path.join(directory_path, "*.avi"))
    avi_files = [path.replace("\\", "/") for path in avi_files]
    
    # Sort the files for consistent ordering
    avi_files.sort()
    
    # Extract timestamps and create tuples
    result = []
    for avi_file in avi_files:
        # Extract filename without extension
        filename = os.path.basename(avi_file)
        filename_no_ext = os.path.splitext(filename)[0]
        
        # Extract timestamp from filename (assuming format: timestamp_picamera.avi)
        # This will extract the part before '_picamera'
        timestamp_match = re.match(r'^(\d{14})_.*', filename_no_ext)
        if timestamp_match:
            timestamp = timestamp_match.group(1)
        else:
            # If timestamp pattern not found, use the full filename without extension
            timestamp = filename_no_ext

        # If filter_timestamp is set, only include matching files using pattern matching
        if filter_timestamp is None or fnmatch.fnmatch(timestamp, filter_timestamp):
            result.append((avi_file, timestamp))
    
    return result

def extract_frames_from_avi_files(avi_files_with_timestamps, base_output_dir="C:/Users/MSAD/github/nnspike/storage/20250824/frames/"):
    """
    Extract frames from all AVI files and save them to folders named by timestamp.
    
    Args:
        avi_files_with_timestamps (list): List of tuples containing (file_path, timestamp)
        base_output_dir (str): Base directory where frame folders will be created
    
    Returns:
        list: List of tuples containing (output_directory, timestamp)
    """
    output_directories_with_timestamps = []
    
    for avi_file, timestamp in avi_files_with_timestamps:
        # Create output directory path
        output_dir = os.path.join(base_output_dir, timestamp)
        
        # Create directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        
        # Add trailing slash for extract_video_frames function
        output_dir_with_slash = output_dir + "/"
        
        filename = os.path.basename(avi_file)
        print(f"Extracting frames from {filename} to {output_dir_with_slash}")
        
        try:
            # Extract frames using the nnspike utility function
            extract_video_frames(avi_file, output_dir_with_slash)
            print(f"✓ Successfully extracted frames to {timestamp}/")
            # Add successful output directory and timestamp to the list
            output_directories_with_timestamps.append((output_dir, timestamp))
        except Exception as e:
            print(f"✗ Error extracting frames from {filename}: {str(e)}")
    
    return output_directories_with_timestamps



In [3]:
# Get all AVI files with their timestamps
avi_files_with_timestamps = get_all_avi_files(directory_path="C:/Users/MSAD/github/nnspike/storage/20250824/videos/")
avi_files_with_timestamps

[('C:/Users/MSAD/github/nnspike/storage/20250824/videos/20250824141051_picamera.avi',
  '20250824141051'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/videos/20250824141133_picamera.avi',
  '20250824141133'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/videos/20250824141159_picamera.avi',
  '20250824141159'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/videos/20250824141226_picamera.avi',
  '20250824141226'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/videos/20250824141308_picamera.avi',
  '20250824141308'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/videos/20250824141526_picamera.avi',
  '20250824141526'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/videos/20250824141713_picamera.avi',
  '20250824141713'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/videos/20250824141857_picamera.avi',
  '20250824141857'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/videos/20250824142217_picamera.avi',
  '20250824142217'),
 ('C:/Users/MSAD/github/nnspike/stora

In [4]:
    # Extract frames from all AVI files
if avi_files_with_timestamps:
    print("\nStarting frame extraction...")
    output_dirs_with_timestamps = extract_frames_from_avi_files(avi_files_with_timestamps, base_output_dir="C:/Users/MSAD/github/nnspike/storage/20250824/frames/")
    print("\nFrame extraction completed!")
    print(f"Successfully created {len(output_dirs_with_timestamps)} output directories:")
    for output_dir, timestamp in output_dirs_with_timestamps:
        print(f"  - {output_dir} (timestamp: {timestamp})")
else:
    print("No AVI files found to process.")
    output_dirs_with_timestamps = []
    
output_dirs_with_timestamps


Starting frame extraction...
Extracting frames from 20250824141051_picamera.avi to C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824141051/
Frames extracted to: C:\Users\MSAD\github\nnspike\storage\20250824\frames\20250824141051
✓ Successfully extracted frames to 20250824141051/
Extracting frames from 20250824141133_picamera.avi to C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824141133/
Frames extracted to: C:\Users\MSAD\github\nnspike\storage\20250824\frames\20250824141133
✓ Successfully extracted frames to 20250824141133/
Extracting frames from 20250824141159_picamera.avi to C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824141159/
Frames extracted to: C:\Users\MSAD\github\nnspike\storage\20250824\frames\20250824141159
✓ Successfully extracted frames to 20250824141159/
Extracting frames from 20250824141226_picamera.avi to C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824141226/
Frames extracted to: C:\Users\MSAD\github\nnspike\storage\

[('C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824141051',
  '20250824141051'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824141133',
  '20250824141133'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824141159',
  '20250824141159'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824141226',
  '20250824141226'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824141308',
  '20250824141308'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824141526',
  '20250824141526'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824141713',
  '20250824141713'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824141857',
  '20250824141857'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824142217',
  '20250824142217'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824142615',
  '20250824142615'),
 ('C:/Users/MSAD/github/nnspike/storage/20250824/frames/2025

In [5]:
for output_dir, timestamp in output_dirs_with_timestamps:
    print(f"Output directory: {output_dir} (timestamp: {timestamp})")
    
    course = "right"
    label_df = create_label_dataframe(output_dir +"/*", course)
    label_df = sort_by_frames_number(label_df)
    label_df = label_dataset_by_opencv(label_df, ROI_CNN, 80)

    status_df = pd.read_csv(f"C:/Users/MSAD/github/nnspike/storage/20250824/sensor_data/{timestamp}_sensor_log.csv")
    df = set_spike_status(label_df, status_df)

    # Export to a csv file
    df.to_csv(f"C:/Users/MSAD/github/nnspike/storage/20250824/labels/{timestamp}_label.csv", index=False)

Output directory: C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824141051 (timestamp: 20250824141051)


Processing: 100%|██████████| 114/114 [00:03<00:00, 31.84it/s]


Output directory: C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824141133 (timestamp: 20250824141133)


Processing: 100%|██████████| 258/258 [00:07<00:00, 36.78it/s]


Output directory: C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824141159 (timestamp: 20250824141159)


...

Processing: 100%|██████████| 1992/1992 [00:50<00:00, 39.13it/s]


Output directory: C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824150405 (timestamp: 20250824150405)


Processing: 100%|██████████| 2047/2047 [00:53<00:00, 38.47it/s]


Output directory: C:/Users/MSAD/github/nnspike/storage/20250824/frames/20250824151025 (timestamp: 20250824151025)


Processing: 100%|██████████| 1603/1603 [00:39<00:00, 40.10it/s]
